<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_04_reductions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-04: Reduction Operations

**Difficulty**: ⬜ Warm-up

**Objective**: Master sum/mean/max/norm along specific dimensions — used in every normalization layer, loss function, and attention mechanism.

In [1]:
import torch

## Task 1: Predict Reduction Shapes

`keepdim=True` keeps the reduced dimension as size 1. This is CRITICAL for broadcasting.

In [2]:
x = torch.randn(2, 3, 4)

# TODO: Predict each shape, then verify
a = x.sum()                           # shape: ???
b = x.sum(dim=-1)                     # shape: ???
c = x.sum(dim=-1, keepdim=True)       # shape: ???
d = x.mean(dim=1)                     # shape: ???
e = x.max(dim=-1)                     # NOTE: returns (values, indices)

print(f'a: {a.shape}')
print(f'b: {b.shape}')
print(f'c: {c.shape}')
print(f'd: {d.shape}')
print(f'e values: {e.values.shape}, indices: {e.indices.shape}')

a: torch.Size([])
b: torch.Size([2, 3])
c: torch.Size([2, 3, 1])
d: torch.Size([2, 4])
e values: torch.Size([2, 3]), indices: torch.Size([2, 3])


## Task 2: Implement Softmax Using Reductions

The numerically stable version: `softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))`

In [16]:
def stable_softmax(x, dim=-1):
    """Implement numerically stable softmax.
    Args:
        x: input tensor of any shape
        dim: dimension to normalize over
    Returns:
        softmax probabilities (same shape as x)
    """
    # TODO: Implement stable softmax
    # Step 1: subtract max for stability (use keepdim=True)
    # Step 2: exponentiate
    # Step 3: divide by sum (use keepdim=True)
    x_max = x.max(dim=dim, keepdim=True)[0]
    x_exp = torch.exp(x - x_max)
    softmax = x_exp / torch.sum(x_exp, dim=dim, keepdim=True)
    return softmax

x = torch.randn(2, 5)
mine = stable_softmax(x)
ref = torch.softmax(x, dim=-1)
assert torch.allclose(mine, ref, atol=1e-6)
# Test with large values (would overflow without stability trick)
x_large = torch.tensor([[1000., 1001., 1002.]])
mine_large = stable_softmax(x_large)
assert not torch.isnan(mine_large).any(), 'NaN detected!'
print('✅ Task 2 passed!')

✅ Task 2 passed!


## Task 3: Implement L2 Norm Along a Dimension

In [24]:
def l2_normalize(x, dim=-1, eps=1e-8):
    """L2 normalize x along given dimension.
    Returns: normalized tensor where each vector along dim has unit length.
    """
    # TODO: Compute L2 norm, then divide
    l2_norm = torch.sqrt(torch.mul(x, x).sum(dim=dim, keepdim=True)) + eps
    x_normalized = x/l2_norm
    return x_normalized


x = torch.randn(3, 4)
normed = l2_normalize(x, dim=-1)
# Each row should have norm = 1
norms = normed.norm(dim=-1)
assert torch.allclose(norms, torch.ones(3), atol=1e-5)
print('✅ Task 3 passed!')

✅ Task 3 passed!
